# Notebook 02 — Exploratory Data Analysis (EDA)
## Social Media Engagement Analytics Dashboard

This notebook provides a comprehensive visual exploration of the cleaned,
feature-engineered YouTube dataset.

### What you will learn
- How metrics are distributed across the dataset
- Which channels, days, and times perform best
- How video duration affects engagement
- Correlations between different metrics

### How to run
1. Run `python main.py run-all` first
2. Then run each cell in order (Shift + Enter)

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))
from src.config import settings

# ── Style configuration ──────────────────────────────────────
plt.rcParams['figure.facecolor'] = '#1A1A2E'
plt.rcParams['axes.facecolor']   = '#16213E'
plt.rcParams['axes.edgecolor']   = '#8E9AAF'
plt.rcParams['axes.labelcolor']  = '#EAEAEA'
plt.rcParams['xtick.color']      = '#EAEAEA'
plt.rcParams['ytick.color']      = '#EAEAEA'
plt.rcParams['text.color']       = '#EAEAEA'
plt.rcParams['grid.color']       = '#2E3A59'
plt.rcParams['grid.alpha']       = 0.5
plt.rcParams['font.size']        = 11

PALETTE = ['#2E86AB', '#F6AE2D', '#4CAF50', '#FF6B35', '#9B59B6',
           '#1ABC9C', '#E74C3C', '#3498DB']

print('✓ Setup complete')

## Section 1 — Dataset Understanding

In [ ]:
csv_path = settings['PROCESSED_DATA_PATH']
if not csv_path.exists():
    print('Processed data not found. Run: python main.py run-all')
else:
    df = pd.read_csv(csv_path, low_memory=False)
    print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')
    print(f'\nColumn names:')
    for c in df.columns:
        print(f'  {c}')

In [ ]:
# Descriptive statistics for numeric columns
numeric_cols = ['view_count','like_count','comment_count',
                'engagement_rate','like_rate','comment_rate',
                'video_duration_minutes','video_age_days']
numeric_cols = [c for c in numeric_cols if c in df.columns]
df[numeric_cols].describe().round(2)

## Section 2 — Univariate Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Metric Distributions', fontsize=16, fontweight='bold', y=1.02)

metrics = [
    ('view_count',           'Views',           PALETTE[0]),
    ('like_count',           'Likes',           PALETTE[1]),
    ('comment_count',        'Comments',        PALETTE[2]),
    ('engagement_rate',      'Engagement Rate %', PALETTE[3]),
    ('video_duration_minutes','Duration (min)', PALETTE[4]),
    ('video_age_days',       'Video Age (days)',PALETTE[5]),
]

for ax, (col, label, color) in zip(axes.flat, metrics):
    if col in df.columns:
        data = df[col].dropna()
        ax.hist(data, bins=30, color=color, edgecolor='none', alpha=0.85)
        ax.set_title(f'{label} Distribution', fontweight='bold')
        ax.set_xlabel(label)
        ax.set_ylabel('Frequency')
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}'))
        median_val = data.median()
        ax.axvline(median_val, color='white', linestyle='--', alpha=0.7,
                   label=f'Median: {median_val:.1f}')
        ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../screenshots/metric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Interpretation: Most metrics are right-skewed — a few viral videos dominate.')
print('This is normal for YouTube data. We use MEDIAN in our recommendations,')
print('not MEAN, because median is more resistant to the effect of viral outliers.')

## Section 3 — Categorical Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Categorical Distribution of Videos', fontsize=16, fontweight='bold')

# Videos by channel
if 'channel_title' in df.columns:
    ch_counts = df['channel_title'].value_counts()
    axes[0,0].barh(ch_counts.index[:10], ch_counts.values[:10], color=PALETTE)
    axes[0,0].set_title('Videos by Channel', fontweight='bold')
    axes[0,0].set_xlabel('Number of Videos')

# Videos by publication day
if 'publication_day_name' in df.columns:
    day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    day_counts = df['publication_day_name'].value_counts().reindex(day_order, fill_value=0)
    axes[0,1].bar(day_counts.index, day_counts.values, color=PALETTE[1])
    axes[0,1].set_title('Videos by Publication Day', fontweight='bold')
    axes[0,1].set_xlabel('Day')
    axes[0,1].set_ylabel('Count')
    axes[0,1].tick_params(axis='x', rotation=30)

# Videos by posting-time group
if 'posting_time_group' in df.columns:
    group_order = ['Late Night','Morning','Afternoon','Evening','Night']
    grp_counts = df['posting_time_group'].value_counts().reindex(group_order, fill_value=0)
    axes[1,0].bar(grp_counts.index, grp_counts.values, color=PALETTE[2])
    axes[1,0].set_title('Videos by Posting-Time Group', fontweight='bold')
    axes[1,0].set_xlabel('Group')
    axes[1,0].tick_params(axis='x', rotation=20)

# Videos by performance category
if 'performance_category' in df.columns:
    perf_order = ['Low','Average','High','Viral']
    perf_colors = ['#E74C3C','#F6AE2D','#4CAF50','#9B59B6']
    perf_counts = df['performance_category'].value_counts().reindex(perf_order, fill_value=0)
    axes[1,1].bar(perf_counts.index, perf_counts.values, color=perf_colors)
    axes[1,1].set_title('Videos by Performance Category', fontweight='bold')
    axes[1,1].set_xlabel('Category')

plt.tight_layout()
plt.savefig('../screenshots/categorical_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 4 — Bivariate Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Bivariate Relationships', fontsize=16, fontweight='bold')

# Views vs Likes
axes[0,0].scatter(df['view_count'], df['like_count'],
                  alpha=0.4, color=PALETTE[0], s=20)
axes[0,0].set_xlabel('Views')
axes[0,0].set_ylabel('Likes')
axes[0,0].set_title('Views vs Likes', fontweight='bold')

# Duration vs Engagement Rate
if 'video_duration_minutes' in df.columns and 'engagement_rate' in df.columns:
    axes[0,1].scatter(df['video_duration_minutes'], df['engagement_rate'],
                      alpha=0.4, color=PALETTE[2], s=20)
    axes[0,1].set_xlabel('Duration (minutes)')
    axes[0,1].set_ylabel('Engagement Rate (%)')
    axes[0,1].set_title('Video Duration vs Engagement Rate', fontweight='bold')

# Publication hour vs Engagement
if 'publication_hour' in df.columns and 'engagement_rate' in df.columns:
    hour_eng = df.groupby('publication_hour')['engagement_rate'].median()
    axes[1,0].plot(hour_eng.index, hour_eng.values, color=PALETTE[1],
                   marker='o', linewidth=2)
    axes[1,0].fill_between(hour_eng.index, hour_eng.values, alpha=0.3, color=PALETTE[1])
    axes[1,0].set_xlabel('Publication Hour (0=midnight, 12=noon)')
    axes[1,0].set_ylabel('Median Engagement Rate (%)')
    axes[1,0].set_title('Posting Hour vs Median Engagement Rate', fontweight='bold')
    axes[1,0].set_xticks(range(0,24,2))

# Channel vs Average Engagement
if 'channel_title' in df.columns and 'engagement_rate' in df.columns:
    ch_eng = df.groupby('channel_title')['engagement_rate'].median().sort_values(ascending=True)
    axes[1,1].barh(ch_eng.index, ch_eng.values, color=PALETTE[3])
    axes[1,1].set_xlabel('Median Engagement Rate (%)')
    axes[1,1].set_title('Channel vs Median Engagement Rate', fontweight='bold')

plt.tight_layout()
plt.savefig('../screenshots/bivariate_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 5 — Correlation Analysis

In [ ]:
corr_cols = [c for c in ['view_count','like_count','comment_count',
             'engagement_rate','video_duration_seconds','video_age_days'] if c in df.columns]

corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels([c.replace('_', '\n') for c in corr_cols], fontsize=9)
ax.set_yticklabels([c.replace('_', '\n') for c in corr_cols], fontsize=9)

# Annotate each cell
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        val = corr_matrix.iloc[i, j]
        text_color = 'white' if abs(val) > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=10, color=text_color, fontweight='bold')

ax.set_title('Correlation Matrix — Engagement Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../screenshots/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('⚠️  IMPORTANT — Correlation ≠ Causation')
print('A high correlation between two metrics means they tend to move together,')
print('NOT that one causes the other.')
print('Example: Views and Likes are correlated because both accumulate over time,')
print('not because having more views forces people to like the video.')

## Section 6 — Best Posting-Time Analysis

In [ ]:
from src.config import settings
MIN_SAMPLE = settings['MIN_SAMPLE_SIZE']

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Posting-Time Analysis (Median Engagement Rate)',
             fontsize=15, fontweight='bold')

# Best posting day
if 'publication_day_name' in df.columns:
    day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    day_stats = df.groupby('publication_day_name').agg(
        n=('video_id','count'),
        median_eng=('engagement_rate','median')
    ).reindex(day_order)
    qualified = day_stats[day_stats['n'] >= MIN_SAMPLE]
    
    colors = ['#F6AE2D' if v == qualified['median_eng'].max() else '#2E86AB'
              for v in qualified['median_eng']]
    axes[0].bar(qualified.index, qualified['median_eng'], color=colors)
    axes[0].set_title('Median Engagement by Publication Day', fontweight='bold')
    axes[0].set_xlabel('Day of Week')
    axes[0].set_ylabel('Median Engagement Rate (%)')
    axes[0].tick_params(axis='x', rotation=30)
    
    best_day = qualified['median_eng'].idxmax()
    print(f'Best posting day (>={MIN_SAMPLE} videos): {best_day}')
    print(f'Median engagement on {best_day}: {qualified.loc[best_day, "median_eng"]:.2f}%')

# Best posting hour
if 'publication_hour' in df.columns:
    hour_stats = df.groupby('publication_hour').agg(
        n=('video_id','count'),
        median_eng=('engagement_rate','median')
    )
    qualified_h = hour_stats[hour_stats['n'] >= MIN_SAMPLE]
    
    axes[1].plot(qualified_h.index, qualified_h['median_eng'],
                 color='#F6AE2D', marker='o', linewidth=2.5)
    axes[1].fill_between(qualified_h.index, qualified_h['median_eng'],
                         alpha=0.25, color='#F6AE2D')
    axes[1].set_title('Median Engagement by Publication Hour', fontweight='bold')
    axes[1].set_xlabel(f'Publication Hour ({settings["REPORTING_TIMEZONE"]})')
    axes[1].set_ylabel('Median Engagement Rate (%)')
    axes[1].set_xticks(range(0,24,2))
    
    best_hour = qualified_h['median_eng'].idxmax()
    print(f'Best posting hour (>={MIN_SAMPLE} videos): {best_hour}:00')
    print(f'Median engagement at hour {best_hour}: {qualified_h.loc[best_hour, "median_eng"]:.2f}%')

plt.tight_layout()
plt.savefig('../screenshots/posting_time_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nNote: Only hours/days with >= {MIN_SAMPLE} videos are shown.')
print('This prevents a single viral video from declaring a time slot as "best".')

## Section 7 — Duration & Category Performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Content Performance Analysis', fontsize=15, fontweight='bold')

# Duration performance
if 'duration_category' in df.columns:
    dur_order = ['Short','Medium','Long']
    dur_eng = df.groupby('duration_category')['engagement_rate'].median().reindex(dur_order)
    axes[0].bar(dur_eng.index, dur_eng.values, color=[PALETTE[0], PALETTE[1], PALETTE[2]])
    axes[0].set_title('Median Engagement by Duration Category', fontweight='bold')
    axes[0].set_xlabel('Duration Category')
    axes[0].set_ylabel('Median Engagement Rate (%)')
    for i, (cat, val) in enumerate(dur_eng.items()):
        axes[0].text(i, val + 0.02, f'{val:.2f}%', ha='center', fontsize=11, fontweight='bold')

# Category performance
if 'category_id' in df.columns:
    cat_eng = df.groupby('category_id')['engagement_rate'].median().sort_values(ascending=True)
    axes[1].barh(cat_eng.index.astype(str), cat_eng.values, color=PALETTE[3])
    axes[1].set_title('Median Engagement by Content Category', fontweight='bold')
    axes[1].set_xlabel('Median Engagement Rate (%)')
    axes[1].set_ylabel('Category ID')

plt.tight_layout()
plt.savefig('../screenshots/content_performance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('=== EDA COMPLETE ===')
print('\nSaved charts to screenshots/')
print('\nNext steps:')
print('1. Run: python main.py analyze  (to generate business_insights.md)')
print('2. Open data/processed/youtube_cleaned_data.csv in Power BI')
print('3. Follow dashboard/power_bi_dashboard_guide.md')